# 05 - ESM-2 protein language model embeddings

This notebook generates frozen, mean-pooled **ESM-2** embeddings (1280-dim) for every protein.

**Why this step:** the Day 5 baselines plateaued at macro-F1 ~0.527 with two
boosting models agreeing to within 0.002 — highlighting that the ceiling is the *feature
representation*, not the classifier. ESM embeddings are a richer representation
and are the natural next move.

**Key discipline:** we apply the **same duplicate-sequence drop** as notebook 04
so the embedding rows line up with the handcrafted-feature rows. The actual
train/test split happens in later notebooks, on whichever feature set is being
evaluated — here we only produce one embedding per (deduplicated) protein.

> Ran this on **Colab with a GPU** (Runtime -> Change runtime type -> GPU).
> The 650M model embeds ~7-8k proteins in a few minutes on a T4.


## 1. Setup

In [ ]:
!pip install -q torch transformers tqdm


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd


from embeddings import ESMEmbedder, save_embeddings, load_embeddings


In [ ]:
# --- Project paths -------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR      = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
EMB_DIR       = DATA_DIR / "embeddings"
SRC_DIR       = PROJECT_ROOT / "src"

EMB_DIR.mkdir(parents=True, exist_ok=True)


sys.path.append(str(SRC_DIR))

## 2. Load the labeled dataset and drop duplicate sequences

Same leakage control as notebook 04: identical sequences must not straddle the
train/test boundary. We dedup *here* so the saved embeddings already reflect the
modeling-ready set of proteins.

In [ ]:
df = pd.read_csv(PROCESSED_DIR / "labeled_dataset.csv")
print(f"Loaded {len(df):,} proteins")

# Drop exact duplicate sequences, keep first occurrence (matches earlier logic).
before = len(df)
df = df.drop_duplicates(subset=["sequence"]).reset_index(drop=True)
print(f"Dropped {before - len(df):,} duplicate sequences -> {len(df):,} unique")

# Sanity check: required columns present
assert {"accession", "sequence"}.issubset(df.columns), "Need accession + sequence columns"
df[["accession", "sequence"]].head()


## 3. Load ESM-2 and embed

`facebook/esm2_t33_650M_UR50D` -> 1280-dim.

In [ ]:
embedder = ESMEmbedder(
    model_name="facebook/esm2_t33_650M_UR50D",
    max_length=1024,   # within ESM-2's 1022-residue limit (+ special tokens)
)
print(f"Model: {embedder.model_name}")
print(f"Device: {embedder.device}")
print(f"Embedding dim: {embedder.dim}")


In [ ]:
embeddings, accessions = embedder.embed_dataframe(
    df,
    id_col="accession",
    seq_col="sequence",
    batch_size=8,      # can be lowered to 4 or 2 if we hit CUDA OOM
)

print("Embeddings shape:", embeddings.shape)   # (n_proteins, 1280)
print("First accession:", accessions[0])


### Sanity checks

In [ ]:
# No NaNs/Infs, row count matches, dim matches the model.
assert np.isfinite(embeddings).all(), "Found NaN/Inf in embeddings"
assert embeddings.shape[0] == len(df) == len(accessions), "Row count mismatch"
assert embeddings.shape[1] == embedder.dim, "Dim mismatch"

print("All checks passed.")
print(f"Mean L2 norm per vector: {np.linalg.norm(embeddings, axis=1).mean():.2f}")


## 4. Save

Stored as `.npy` + an aligned `metadata.csv` (row order = accession). 
A vector database is intentionally *not* used here: the full matrix is small (~40 MB) and every
downstream step wants it in memory at once.

In [ ]:
emb_path, meta_path = save_embeddings(embeddings, accessions, EMB_DIR)
print(f"Saved embeddings -> {emb_path}")
print(f"Saved metadata   -> {meta_path}")

size_mb = emb_path.stat().st_size / 1e6
print(f"Embedding file size: {size_mb:.1f} MB")


In [ ]:
# Round-trip check: reload and confirm it matches.
emb2, meta2 = load_embeddings(EMB_DIR)
assert np.allclose(emb2, embeddings)
assert list(meta2["accession"]) == [str(a) for a in accessions]
print("Round-trip load OK. Ready for notebook 06 (embedding + hybrid models).")
